# MOM6 Sea Surface Temperature Subsetting
Author: Brooke Hawkins

I'm creating a notebook to process MOM6 data in order to recreate the Sea Surface Temperature (SST) anomaly plot that Isaac Schroeder produced for the 2025 mini ESR.

Currently, the plot is produced with [OISST](https://www.ncei.noaa.gov/products/optimum-interpolation-sst). Here's a brief description of that dataset: "The Optimum Interpolation Sea Surface Temperature (OISST) dataset is a global sea surface temperature dataset. Data is on a regular grid at ¼ degree intervals, from September 1, 1981 to present. Temperature data comes from satellites, ships, buoys and Argo floats; temperature is interpolated in between observations."

Here's an example of the plot, for Spring 2025: [oc_miniESR_sst_mons_4_6_contour.png](https://github.com/id-sch/oc_cciea.github.io/blob/main/figures_gha/miniESR/2025/sst_anom/oc_miniESR_sst_mons_4_6_contour.png). The figure is plotted for 32-48 °N, 128-116.8 °W, at 1/2 degree resolution. The anomalies are calculated using a climatological mean calculated from monthly data 1982 - current, and subtracting that mean from seasonal mean (e.g. April to June, 2025 for spring 2025).

Here's the repository from Isaac: https://github.com/id-sch/oc_cciea.github.io. There are two key scripts he uses to produce the plot.
1. [sst_oi_daily_v21_update.py](https://github.com/id-sch/oc_cciea.github.io/blob/main/code_gha/SSTspatial/sst_oi_daily_v21_update.py) downloads OISST data every month using ERDDAP, and saves it in a netCDF.
2. [miniESR/countour_noaaoisst_cilm_anoms_season.py](https://github.com/id-sch/oc_cciea.github.io/blob/main/code_gha/miniESR/contour_noaaoisst_clim_anoms_season.py) calculates the climatological and seasonal means, calculates the anomalies, then plots the anomalies along the US West Coast.

I'm going to recreate the same plot using MOM6-NEP10k, a version of Modular Ocean Model 6 (MOM6) for the Northeast Pacific (NEP), available on the [CEFI data portal](https://psl.noaa.gov/cefi_portal/#data_access). The regirdded sea surface temperature data is stored in variable name *tos*. The most recent hindcast release is version *r20250912*, and has an available date range is 1993 to 2025. At time of writing, the forecast is not available on the portal for NEP. Instead of downloading from the THREDDS server, I'm going to use the demo [Earthmover platform](https://app.earthmover.io/NOAA-PMEL).

I developed this code, with syntax formatting and debugging assistance using Google Gemini.

In [1]:
import xarray as xr
import numpy as np
import pandas as pd
import datetime
import time
from arraylake import Client
from dask.diagnostics import ProgressBar

In [2]:
# Print the date and time this notebook is run
run_time = datetime.datetime.now().strftime("%B %d, %Y at %I:%M %p")
print(f"Data retrieved and processed on: {run_time}")

# initialize connection to data repository on Earthmover
client = Client()
repo = client.get_repo("NOAA-PMEL/cefi-nep-hindcast-monthly")

# open read-only session pointing to main branch
session = repo.readonly_session(branch="main")

# lazy load the zarr data store
aux = xr.open_zarr(session.store, zarr_format=3, group="regrid/aux1")

# extract sea surface tempreature variable
ds_temp = aux[["tos"]]

# print dataset info to check dimensions and coordinates
print(ds_temp)

# define spatial extent for California Current
x_lim = [-128, -116.8]
y_lim = [32, 48]

# subset the dataset (add 360 to longitude for 0-360 coordinate scale)
ds_subset = ds_temp.sel(
    lat=slice(y_lim[0], y_lim[1]), lon=slice(x_lim[0] + 360, x_lim[1] + 360)
)

# print estimated dataset size before saving
print(f"Estimated download size: {ds_subset.nbytes / 1e9:.2f} GB")

# reformat the dataset to match Isaac's output from OISST by
# 1. rename spatial dimensions to match downstream script
# 2. shift timestamps to the first of the month at midnight
ds_formatted = ds_subset.rename({"lat": "lat_vec", "lon": "lon_vec"})
ds_formatted["time"] = (
    ds_formatted["time"].astype("datetime64[M]").astype("datetime64[ns]")
)

# double check coordinates before saving
print(ds_formatted.coords)

# write subset to netCDF and track download with progress bar
with ProgressBar():
    ds_formatted.to_netcdf("mom6_monthly_sst.nc")

Data retrieved and processed on: July 13, 2026 at 10:35 PM
<xarray.Dataset> Size: 440MB
Dimensions:  (time: 396, lat: 815, lon: 341)
Coordinates:
  * time     (time) datetime64[ns] 3kB 1993-01-16T12:00:00 ... 2025-06-16
  * lat      (lat) float64 7kB 10.81 10.89 10.98 11.07 ... 80.55 80.63 80.72
  * lon      (lon) float64 3kB 156.9 157.2 157.5 157.8 ... 254.4 254.7 255.0
Data variables:
    tos      (time, lat, lon) float32 440MB dask.array<chunksize=(100, 200, 200), meta=np.ndarray>
Attributes: (12/26)
    NumFilesInSet:          1
    title:                  NEP10k_202507_physics_bgc
    grid_type:              regular
    grid_tile:              N/A
    NCO:                    netCDF Operators version 5.2.4 (Homepage = http:/...
    cefi_rel_path:          cefi_portal/northeast_pacific/full_domain/hindcas...
    ...                     ...
    cefi_init_date:         N/A
    cefi_ensemble_info:     N/A
    cefi_forcing:           N/A
    cefi_data_doi:          10.5281/zenodo.139362

Once I had the netCDF saved with monthly MOM6 data, I made a couple minor edits to Isaac's visualization code:
* I changed the `yr_bgn` to 1993 instead of the first year in the time series, and `iea_yr` to 2025 instead of 2026, to match the timeframe available for the MOM6 NEP hindcast.
* I changed the file name of the netCDF from `TS_monthly.nc` to `mom6_monthly_sst.nc`, and I changed the `var_wnt` variable from `sst` to `tos` to match the MOM6 file I saved.
* I changed the output plot directory `dir_plot_out` to './' instead of using the 'figures_gha/mini_esr/sst_anom' structure from Isaac's repository, since I didn't copy the folder structure in this repository.